# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>Table of Contents</b></p></div>
    
- [Introduction](#1)
- [Explore](#2)
- [Clustering Tendency](#3)
- [Data Transformation - PCA](#4)
- [K-Means](#5)
- [K-Means PCA](#6)
- [Agglomerative Clustering](#7)
- [Spectral Clustering](#8)
- [Visualization](#9)
- [Conclusion](#10)


# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>Introduction</b></p></div>

This study evaluates the abalone dataset and attempts to apply a clustering technique to see whether the predictive models can be improved with the clusters. This study finds that if you apply good clustering you can indeed improve the performance of a predictive model.

In [ ]:
import numpy as np
import pandas as pd
from pandas_profiling import ProfileReport

from IPython.core.display import display, HTML

import seaborn as sns
import matplotlib.pyplot as plt
import warnings

# Configure Jupyter Notebook
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 500) 
pd.set_option('display.expand_frame_repr', False)
display(HTML("<style>div.output_scroll { height: 35em; }</style>"))

%matplotlib inline
%config InlineBackend.figure_format ='retina'

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/kaggle/input/wine-dataset-for-clustering/wine-clustering.csv')


# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>Explore</b></p></div>

To start the exploration the data quality is checked

In [ ]:
df.info()

This tells us that there is no missing values

In [ ]:
df.head(10)

This gives us a glimpse of the data where we can see mostly numerical features with one categorical feature

In [ ]:
df.describe(include='all').T

The descriptive features above shows us the distribution of the values, where we can see a few outliers in the numerical data (for instance the maximum for Height and shucked weight). We can apply clamping, or simply use models/methods that are less sensitive to outliers. For instance scaling using a standard scaler and modelling using something like a random forest. 

In [ ]:
%%time
profile = ProfileReport(df,
                        title="Profiling Report",
                        dataset={"description": "This profiling report was generated for Carl Kirstein",
                                 "copyright_holder": "Carl Kirstein",
                                 "copyright_year": "2022",
                                },
                        explorative=True,
                       )
profile
# profile.to_file("Tutorial 2.html")

The profiling report above shows us that the data quality is not problematic. 

In [ ]:
plt.figure(figsize=(10,10))
sns.set_style('white')
plot_kws={"s": 1}
g = sns.pairplot(
             df,
             diag_kind='hist',
             corner=False,
             palette='cividis',
            )

plt.show()

The pairplot (SPLOM) above shows us that there are not clear clusters at a first glance. We are looking for clusters, so this will require us to transform the data. PCA is immediately considered. 

We also see that it is not easy to distinguish the number of rings simply through correlation to any single features, although there is a suggestion that the age of the abalone is related to its size. 

In [ ]:
plt.figure(figsize=(10,10))
threshold = 0.5
sns.set_style("whitegrid", {"axes.facecolor": ".0"})
df_cluster2 = df.corr()
mask = df_cluster2.where((abs(df_cluster2) >= threshold)).isna()
plot_kws={"s": 1}
sns.heatmap(df_cluster2,
            cmap='RdYlBu',
            annot=True,
            mask=mask,
            linewidths=0.2, 
            linecolor='lightgrey').set_facecolor('white')

The correlation heatmap above shows us that there are highly correlated features, i.e. length<->diameter and whole weight -> shucked weight - viscera weight - shell-weight. 

Diameter and whole weight can then be dropped.

In [ ]:
%%time

def corrdot(*args, **kwargs):
    corr_r = args[0].corr(args[1])
    corr_text = f"{corr_r:2.2f}".replace("0.", ".")
    ax = plt.gca()
    ax.set_axis_off()
    marker_size = abs(corr_r) * 10000
    ax.scatter([.5], [.5], marker_size, [corr_r], alpha=0.6, cmap='coolwarm',
               vmin=-1, vmax=1, transform=ax.transAxes)
    font_size = abs(corr_r) * 40 + 5
    ax.annotate(corr_text, [.5, .5,],  xycoords="axes fraction",
                ha='center', va='center', fontsize=font_size)

sns.set(style='white', font_scale=1.6)
g = sns.PairGrid(df, aspect=1.4, diag_sharey=False)
g.map_lower(sns.regplot, lowess=True, ci=False, line_kws={'color': 'black','lw': 1.5}, scatter_kws={'s':8,'alpha':0.5,'color':'gray'})
g.map_diag(sns.distplot, kde_kws={'color': 'black'},hist_kws={'color':'gray','alpha':1,})
g.map_upper(corrdot)

The correlation reg-plot above gives us a nicer view of the correlations and the structure of the data in one.

# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>Clustering Tendency</b></p></div>

The next step in the quest to find clusters will be to apply clustering tendency evaluations. 
We start with the Hopkins statistic that tells us that there's clusters when the value is >0.75

In [ ]:
from random import sample
from numpy.random import uniform
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import scale

# function to compute hopkins's statistic for the dataframe X
def hopkins_statistic(X,k=2):
    
    X=X.values  #convert dataframe to a numpy array
    X = scale(X)
    sample_size = int(X.shape[0]*0.05) #0.05 (5%) based on paper by Lawson and Jures
    
    
    #a uniform random sample in the original data space
    X_uniform_random_sample = uniform(X.min(axis=0), X.max(axis=0) ,(sample_size , X.shape[1]))
    
    
    
    #a random sample of size sample_size from the original data X
    random_indices=sample(range(0, X.shape[0], 1), sample_size)
    X_sample = X[random_indices]
   
    
    #initialise unsupervised learner for implementing neighbor searches
    neigh = NearestNeighbors(n_neighbors=k)
    nbrs=neigh.fit(X)
    
    #u_distances = nearest neighbour distances from uniform random sample
    u_distances , u_indices = nbrs.kneighbors(X_uniform_random_sample , n_neighbors=k)
    u_distances = u_distances[: , 0] #distance to the first (nearest) neighbour
    
    #w_distances = nearest neighbour distances from a sample of points from original data X
    w_distances , w_indices = nbrs.kneighbors(X_sample , n_neighbors=k)
    #distance to the second nearest neighbour (as the first neighbour will be the point itself, with distance = 0)
    w_distances = w_distances[: , 1]
    
 
    
    u_sum = np.sum(u_distances)
    w_sum = np.sum(w_distances)
    
    #compute and return hopkins' statistic
    H = u_sum/ (u_sum + w_sum)
    return H
    

In [ ]:
df.columns

In [ ]:
df = pd.get_dummies(df)

In [ ]:
from sklearn.preprocessing import StandardScaler
float_columns = [x for x in df.columns]
sc = StandardScaler()
df2 = df.copy()
df[float_columns] = sc.fit_transform(df[float_columns])
df.head()

In [ ]:
X = df

In [ ]:
# call the function on the iris dataset
H=hopkins_statistic(X,3)
print('Hopkins statistic: '+str(H))

The Hopkins statistic is not greater than 0.75, but it is close to it, so we have a suspision that there could be clusters. 

In [ ]:
!pip uninstall pyclustertend -y
!pip install pyclustertend

The next step is a VAT where we check whether there are clustering tendencies. 

In [ ]:
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import pairwise_distances
from numba import njit


def vat(data: np.ndarray, return_odm: bool = False, figure_size: Tuple = (10, 10)):
    """VAT means Visual assessment of tendency. basically, it allow to asses cluster tendency
    through a map based on the dissimilarity matrix.
    Parameters
    ----------
    data : matrix
        numpy array
    return_odm : return the Ordered dissimilarity Matrix
        boolean (default to False)
    figure_size : size of the VAT.
        tuple (default to (10,10))
    Return
    -------
    ODM : matrix
        the ordered dissimilarity matrix plotted.
    """

    ordered_dissimilarity_matrix = compute_ordered_dissimilarity_matrix(data)

    _, ax = plt.subplots(figsize=figure_size)
    ax.imshow(
        ordered_dissimilarity_matrix,
        cmap="coolwarm",
        vmin=0,
        vmax=np.max(ordered_dissimilarity_matrix),
    )

    if return_odm is True:
        return ordered_dissimilarity_matrix


@njit(cache=True)
def compute_ordered_dis_njit(matrix_of_pairwise_distance: np.ndarray):  # pragma: no cover
    """
    The ordered dissimilarity matrix is used by visual assessment of tendency. It is a just a a reordering
    of the dissimilarity matrix.
    Parameter
    ----------
    x : matrix
        numpy array
    Return
    -------
    ODM : matrix
        the ordered dissimilarity matrix
    """

    # Step 1 :

    observation_path = np.zeros(matrix_of_pairwise_distance.shape[0], dtype="int")

    list_of_int = np.zeros(matrix_of_pairwise_distance.shape[0], dtype="int")

    index_of_maximum_value = np.argmax(matrix_of_pairwise_distance)

    column_index_of_maximum_value = (
        index_of_maximum_value // matrix_of_pairwise_distance.shape[1]
    )

    list_of_int[0] = column_index_of_maximum_value
    observation_path[0] = column_index_of_maximum_value

    K = np.linspace(
        0,
        matrix_of_pairwise_distance.shape[0] - 1,
        matrix_of_pairwise_distance.shape[0],
    ).astype(np.int32)

    J = np.delete(K, column_index_of_maximum_value)

    for r in range(1, matrix_of_pairwise_distance.shape[0]):

        p, q = (-1, -1)

        mini = np.max(matrix_of_pairwise_distance)

        for candidate_p in observation_path[0:r]:
            for candidate_j in J:
                if matrix_of_pairwise_distance[candidate_p, candidate_j] < mini:
                    p = candidate_p
                    q = candidate_j
                    mini = matrix_of_pairwise_distance[p, q]

        list_of_int[r] = q
        observation_path[r] = q
        ind_q = np.where(J == q)[0][0]
        J = np.delete(J, ind_q)

    # Step 3

    ordered_matrix = np.zeros(matrix_of_pairwise_distance.shape)

    for column_index_of_maximum_value in range(ordered_matrix.shape[0]):
        for j in range(ordered_matrix.shape[1]):
            ordered_matrix[
                column_index_of_maximum_value, j
            ] = matrix_of_pairwise_distance[
                list_of_int[column_index_of_maximum_value], list_of_int[j]
            ]

    # Step 4 :

    return ordered_matrix


def compute_ordered_dissimilarity_matrix(x: np.ndarray) -> np.ndarray:
    matrix_of_pairwise_distance = pairwise_distances(x)
    dis_matrix = compute_ordered_dis_njit(matrix_of_pairwise_distance)
    return dis_matrix


def ivat(data: np.ndarray, return_odm: bool = False, figure_size: Tuple = (10, 10)):
    """iVat return a visualisation based on the Vat but more reliable and easier to
    interpret.
    Parameters
    ----------
    data : matrix
        numpy array
    return_odm : return the Ordered dissimilarity Matrix
            boolean (default to False)
    figure_size : size of the VAT.
        tuple (default to (10,10))
    Return
    -------
    D_prim : matrix
        the ivat ordered dissimilarity matrix
    """

    ordered_matrix = compute_ivat_ordered_dissimilarity_matrix(data)

    _, ax = plt.subplots(figsize=figure_size)
    ax.imshow(ordered_matrix, cmap="coolwarm", vmin=0, vmax=np.max(ordered_matrix))

    if return_odm is True:
        return ordered_matrix


def compute_ivat_ordered_dissimilarity_matrix(x: np.ndarray):
    """The ordered dissimilarity matrix is used by ivat. It is a just a a reordering
    of the dissimilarity matrix.
    Parameters
    ----------
    x : matrix
        numpy array
    Return
    -------
    D_prim : matrix
        the ordered dissimilarity matrix
    """

    ordered_matrix = compute_ordered_dissimilarity_matrix(x)
    re_ordered_matrix = np.zeros((ordered_matrix.shape[0], ordered_matrix.shape[0]))

    for r in range(1, ordered_matrix.shape[0]):
        # Step 1 : find j for which D[r,j] is minimum and j ipn [1:r-1]

        j = np.argmin(ordered_matrix[r, 0:r])

        # Step 2 :

        re_ordered_matrix[r, j] = ordered_matrix[r, j]
        re_ordered_matrix[j, r] = ordered_matrix[r, j]

        # Step 3 : for c : 1, r-1 with c !=j
        c_tab = np.array(range(0, r))
        c_tab = c_tab[c_tab != j]

        for c in c_tab:
            re_ordered_matrix[r, c] = max(ordered_matrix[r, j], re_ordered_matrix[j, c])
            re_ordered_matrix[c, r] = re_ordered_matrix[r, c]

    return re_ordered_matrix

First the data is scaled through standardization

Then the VAT is performed

In [ ]:
X=np.array(df)

In [ ]:
%%time
vat(X)

So the VAT confirms that there are clusters. And it seems that there are three distinct clusters in there somewhere. So the next sections will be to see whether we can get those distinct clusters. 

# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>Data Transformation - PCA</b></p></div>

We saw previously with the pairplot and correlation plot that the clusters are not distinct. Therefore a PCA is done.

In [ ]:
from sklearn.decomposition import PCA

# Create principal components
pca = PCA(n_components=5)   # limit the number of principal components to spare the pairplots
X_pca = pca.fit_transform(df)

# Convert to dataframe
component_names = [f"PC{i+1}" for i in range(X_pca.shape[1])]
X_pca = pd.DataFrame(X_pca, columns=component_names)

X_pca.head(10)

When the pairplot (SPLOM) is repeated for the Princpal Components in the figure below, the three clusters appears at PC1 + PC2.

In [ ]:
plt.figure(figsize=(15,15))
sns.set_style('white')
plot_kws={"s": 1}
g = sns.pairplot(
             X_pca,
             diag_kind='hist',
             corner=False,
             palette='cividis',
            )

plt.show()

The shape for PC2 is familiar when visiting https://scikit-learn.org/stable/modules/clustering.html

In [ ]:
def plot_variance(pca, width=8, dpi=100):
    # Create figure
    fig, axs = plt.subplots(1, 2)
    n = pca.n_components_
    grid = np.arange(1, n + 1)
    # Explained variance
    evr = pca.explained_variance_ratio_
    axs[0].bar(grid, evr)
    axs[0].set(
        xlabel="Component", title="% Explained Variance", ylim=(0.0, 1.0)
    )
    # Cumulative Variance
    cv = np.cumsum(evr)
    axs[1].plot(np.r_[0, grid], np.r_[0, cv], "o-")
    axs[1].set(
        xlabel="Component", title="% Cumulative Variance", ylim=(0.0, 1.0)
    )
    # Set up figure
    fig.set(figwidth=8, dpi=100)
    return axs

In [ ]:
plt.style.use('seaborn-white')
plt.rcParams['figure.figsize']=5,5 
plt.rcParams['font.family'] = 'Calibri'
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 1
plt.rcParams['axes.labelsize']=12
plt.rcParams['xtick.labelsize']=12
plt.rcParams['ytick.labelsize']=12
plt.rcParams['legend.fontsize']=12
plot_variance(pca);

In [ ]:
pca.explained_variance_ratio_

The explained variance ratio was chosen to contribute more than 6%. 

In [ ]:
# component loadings or weights (correlation coefficient between original variables and the component) 
# component loadings represents the elements of the eigenvector
# the squared loadings within the PCs always sums to 1
loadings = pca.components_
num_pc = pca.n_features_
pc_list = ["PC"+str(i) for i in list(range(1, num_pc+1))]
loadings_df = pd.DataFrame.from_dict(dict(zip(pc_list, loadings)))
loadings_df['variable'] = df.columns.values
loadings_df = loadings_df.set_index('variable')
loadings_df

In [ ]:
plt.figure(figsize=(10,10))
threshold = 0.
sns.set_style("whitegrid", {"axes.facecolor": ".0"})
df_cluster2 = loadings_df
mask = df_cluster2.where((abs(df_cluster2) >= threshold)).isna()
plot_kws={"s": 1}
sns.heatmap(df_cluster2,
            cmap='RdYlBu',
            annot=True,
            mask=mask,
            linewidths=0.2, 
            linecolor='lightgrey').set_facecolor('white')

PC1 is affected by many dimensions. Whereas PC2 is affected mostly by alchol and color intensity. PC3 is affected most by the ash features. 


# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>K-Means</b></p></div>

The go to clustering method is kmeans, although it is probably not fully suited to the cluster shapes observed thus far. It should be informative nonetheless. The first k-means application is without the principal components. We'll start with the 3 clusters, but we'll shows throughout this notebook why 3 cluster have been selected. 

In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans = kmeans.fit(df)

In [ ]:
df['kmeans'] = kmeans.predict(df[float_columns])

In [ ]:
# create and fit a range of models: 
km_list = list()
for clust in range(1, 21):
    km = KMeans(n_clusters=clust, random_state=42)
    km = km.fit(df)
    km_list.append(pd.Series({'clusters': clust,
                             'inertia':km.inertia_,
                             'model':km}))

In [ ]:
plt.style.use('seaborn-white')
plt.rcParams['figure.figsize']=5,5 
plt.rcParams['font.family'] = 'Calibri'
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 0.5
plt.rcParams['axes.labelsize']=12
plt.rcParams['xtick.labelsize']=12
plt.rcParams['ytick.labelsize']=12
plt.rcParams['legend.fontsize']=12

plot_data = (pd.concat(km_list, axis=1)
             .T
             [['clusters','inertia']]
             .set_index('clusters'))

ax = plot_data.plot(marker='o', ls='-', color='steelblue')
ax.set_title('K-means')
ax.set_xticks(range(0, 21, 2))
ax.set_xlim(0,21)
ax.set(xlabel='Cluster', ylabel='Inertia');

The elbow method is attempted and it is clearly at 3 clusters.

# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>K-Means PCA</b></p></div>

the previous section did k-means on the orginal features, this section focuses more on the principal components. 

In [ ]:
from sklearn.cluster import KMeans
kmeansPCA = KMeans(n_clusters=3, random_state=42)
kmeansPCA = kmeansPCA.fit(X_pca)
X_pca['kmeans PCA'] = kmeansPCA.labels_
X_pca.head()

In [ ]:
df2 = pd.concat([df.reset_index(drop=True),X_pca],axis=1)
df2.head()

In [ ]:
df['kmeans PCA'] = df2['kmeans PCA']

In [ ]:
plt.figure(figsize=(15,15))
sns.set_style('white')
plot_kws={"s": 1}
g = sns.pairplot(
             df2.drop(['kmeans'],axis=1),
             diag_kind='hist',
             corner=False,
             palette='cividis',
             hue='kmeans PCA'
            )

plt.show()

Okay, it turns out when checking PC1 that k-means picks up the clusters quite well. Although we're going to try and do even better in the rest of this notebook. 

In [ ]:
# create and fit a range of models: 
km_list = list()
for clust in range(1, 21):
    kmPCA2 = KMeans(n_clusters=clust, random_state=42)
    kmPCA2 = kmPCA2.fit(X_pca)
    km_list.append(pd.Series({'clusters': clust,
                             'inertia':kmPCA2.inertia_,
                             'model':kmPCA2}))

In [ ]:
plt.style.use('seaborn-white')
plt.rcParams['figure.figsize']=5,5 
plt.rcParams['font.family'] = 'Calibri'
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 0.5
plt.rcParams['axes.labelsize']=12
plt.rcParams['xtick.labelsize']=12
plt.rcParams['ytick.labelsize']=12
plt.rcParams['legend.fontsize']=12

plot_data = (pd.concat(km_list, axis=1)
             .T
             [['clusters','inertia']]
             .set_index('clusters'))

ax = plot_data.plot(marker='o', ls='-', color='steelblue')
ax.set_title('K-means with PCA')
ax.set_xticks(range(0, 21, 2))
ax.set_xlim(0,21)
ax.set(xlabel='Cluster', ylabel='Inertia');

What we do find with the principal components identified, the elbow is also clearly at 3 clusters. 

# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>Agglomerative Clustering</b></p></div>

a cluster method that could work on the set is agglomerative clustering. The three clusters identified thus far is used for the analysis.  

In [ ]:
from sklearn.cluster import AgglomerativeClustering
ag = AgglomerativeClustering(n_clusters=3, linkage='ward', compute_full_tree=True)
ag = ag.fit(df)
df['agglom'] = ag.fit_predict(df[float_columns])

In [ ]:
plt.style.use('seaborn-white')
plt.rcParams['figure.figsize']=8,8 
plt.rcParams['font.family'] = 'Calibri'
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 1
plt.rcParams['axes.labelsize']=12
plt.rcParams['xtick.labelsize']=12
plt.rcParams['ytick.labelsize']=12
plt.rcParams['legend.fontsize']=12

from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering


def plot_dendrogram(model, **kwargs):
    # Create linkage matrix and then plot the dendrogram

    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    # Plot the corresponding dendrogram
    dendrogram(linkage_matrix, **kwargs)


# setting distance_threshold=0 ensures we compute the full tree.
model = AgglomerativeClustering(linkage='ward',distance_threshold=0, n_clusters=None)

model = model.fit(df[float_columns])
plt.title("Hierarchical Clustering Dendrogram")


plot_dendrogram(model, truncate_mode="level", p=5)
plt.xlabel("Number of points in node (or index of point if no parenthesis).")
sns.despine()
plt.show()

The chart above gives us a nice view of how to separate the clusters. One could argue that the seperation at three clusters is a feasible place, as the number of clusters rapidly increase with diminishing inter cluster distances. 

# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>Spectral Clustering</b></p></div>

In [ ]:
from sklearn.cluster import SpectralClustering
clustering = SpectralClustering(n_clusters=3,
                                assign_labels='discretize',
                                random_state=0).fit(X_pca)
X_pca['spectral'] = clustering.labels_

In [ ]:
df['spectral'] = X_pca['spectral']

In [ ]:
plt.figure(figsize=(15,15))
sns.set_style('white')
plot_kws={"s": 1}
g = sns.pairplot(
             X_pca.drop(['kmeans PCA'],axis=1),
             # kind='reg',
             diag_kind='hist',
             corner=False,
             # plot_kws=dict(scatter_kws=dict(s=5)),
             palette='cividis',
             hue='spectral'
            )

# g = g.map_diag(sns.kdeplot, lw=2)
# g = g.map_offdiag(sns.kdeplot, lw=0.5)
plt.show()

The spectral clustering is not apparently picking up the clusters well at the intersection of PC1 ad PC2. 

# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>Visualization</b></p></div>

Ini this section we investigate the clustering more visually through SPLOMs and parallel coordinate plots

## <div style="color:white;display:fill;border-radius:5px;background-color:#aaaaaa;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>SPLOM</b></p></div>

In [ ]:
df2['agglom']=df['agglom']
df2['spectral']=df['spectral']

In [ ]:
plt.figure(figsize=(15,15))
sns.set_style('white')
plot_kws={"s": 1}
g = sns.pairplot(
             df.drop(['kmeans','agglom','spectral'],axis=1),
             # kind='reg',
             diag_kind='hist',
             corner=False,
             # plot_kws=dict(scatter_kws=dict(s=5)),
             palette='cividis',
             hue='kmeans PCA'
            )

# g = g.map_diag(sns.kdeplot, lw=2)
# g = g.map_offdiag(sns.kdeplot, lw=0.5)
plt.show()

In [ ]:
plt.figure(figsize=(15,15))
sns.set_style('white')
plot_kws={"s": 1}
g = sns.pairplot(
             df.drop(['kmeans','kmeans PCA','spectral'],axis=1),
             # kind='reg',
             diag_kind='hist',
             corner=False,
             # plot_kws=dict(scatter_kws=dict(s=5)),
             palette='cividis',
             hue='agglom'
            )

# g = g.map_diag(sns.kdeplot, lw=2)
# g = g.map_offdiag(sns.kdeplot, lw=0.5)
plt.show()

## <div style="color:white;display:fill;border-radius:5px;background-color:#aaaaaa;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>Parallel Coordinate Plot</b></p></div>


In [ ]:
df2['kmeans'] = df['kmeans'].astype(str)
df2['agglom'] = df['agglom'].astype(str)

In [ ]:
plt.style.use('seaborn-white')
plt.rcParams['figure.figsize']=20,10 
plt.rcParams['font.family'] = 'Calibri'
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 0.5
plt.rcParams['axes.labelsize']=12
plt.rcParams['xtick.labelsize']=12
plt.rcParams['ytick.labelsize']=12
plt.rcParams['legend.fontsize']=12

from pandas.plotting import parallel_coordinates

parallel_coordinates(df2.drop(['kmeans','agglom','spectral'],axis=1), 
                     'kmeans PCA',
                    colormap='cividis')
plt.xticks(rotation=45)
plt.show()

Kmeans does quite well, seperating the clusters on Total_phenols, Flavanoids, and PC1. 

In [ ]:
plt.style.use('seaborn-white')
plt.rcParams['figure.figsize']=20,10 
plt.rcParams['font.family'] = 'Calibri'
plt.rcParams['font.size'] = 10
plt.rcParams['lines.linewidth'] = 0.5
plt.rcParams['axes.labelsize']=10
plt.rcParams['xtick.labelsize']=10
plt.rcParams['ytick.labelsize']=10
plt.rcParams['legend.fontsize']=10

from pandas.plotting import parallel_coordinates

# data = pandas.read_csv(r'C:\Python27\Lib\site-packages\pandas\tests\data\iris.csv', sep=',')
parallel_coordinates(df2.drop(['kmeans','kmeans PCA','agglom'],axis=1), 
                     'spectral',
                    colormap='cividis')

plt.xticks(rotation=45)
plt.show()

spectral clustering is clearly not doing a great job. 

In [ ]:
plt.style.use('seaborn-white')
plt.rcParams['figure.figsize']=20,10 
plt.rcParams['font.family'] = 'Calibri'
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 0.5
plt.rcParams['axes.labelsize']=12
plt.rcParams['xtick.labelsize']=12
plt.rcParams['ytick.labelsize']=12
plt.rcParams['legend.fontsize']=12

from pandas.plotting import parallel_coordinates
parallel_coordinates(df2.drop(['kmeans','spectral','kmeans PCA'],axis=1), 
                     'agglom',
                    colormap='cividis')
plt.xticks(rotation=45)
plt.show()

Agglomerative clustering is doing quite well, seperating the clusters on Total_phenols, Flavanoids, and PC1. 

# <div style="color:white;display:fill;border-radius:5px;background-color:#555555;font-family:Arial"><p style="padding: 8px;color:white;font-size:120%"><b>Conclusion</b></p></div>

The conclusion is that there are three clearly defined clusters that become more visible with a PCA. Kmeans clustering did quite well on the PCA data, and the agglomerative clustering did well considering that it was not done on the PCA data, but on the original data set only. 